# Org1 Graph Construction

In [1]:
from pathlib import Path
import duckdb

In [2]:
ORG1_PATH = (
    Path("../data/organizations/organization=Org1")
    .resolve()
    .as_posix()
    + "/*.parquet"
)

print(ORG1_PATH)

C:/Users/Gulso/Documents/Gul/CredBlock-FL/data/organizations/organization=Org1/*.parquet


In [3]:
org1_users = duckdb.sql(f"""
    SELECT
        "User ID",
        ROW_NUMBER() OVER (
            ORDER BY "User ID"
        ) - 1 AS user_node_id

    FROM (
        SELECT DISTINCT
            "User ID"

        FROM read_parquet(
            '{ORG1_PATH}'
        )
    )

    ORDER BY user_node_id
""").df()

print(org1_users.shape)
display(org1_users.head())

(791567, 2)


,User ID,user_node_id
0,-9223371191532286299,0
1,-9223351024921207274,1
2,-9223320431696609944,2
3,-9223311529942795212,3
4,-9223287066183308537,4


In [4]:
org1_ips = duckdb.sql(f"""
    SELECT
        "IP Address",
        ROW_NUMBER() OVER (
            ORDER BY "IP Address"
        ) - 1 AS ip_node_id

    FROM (
        SELECT DISTINCT
            "IP Address"

        FROM read_parquet(
            '{ORG1_PATH}'
        )
    )

    ORDER BY ip_node_id
""").df()

print(org1_ips.shape)
display(org1_ips.head())

(733381, 2)


,IP Address,ip_node_id
0,10.0.0.1,0
1,10.0.0.10,1
2,10.0.0.100,2
3,10.0.0.101,3
4,10.0.0.102,4


In [5]:
USER_COUNT = len(org1_users)

org1_ips["global_ip_node_id"] = (
    org1_ips["ip_node_id"] + USER_COUNT
)

display(org1_ips.head())

print("User node ID range:")
print(org1_users["user_node_id"].min(), "to",
      org1_users["user_node_id"].max())

print("\nIP node ID range:")
print(org1_ips["global_ip_node_id"].min(), "to",
      org1_ips["global_ip_node_id"].max())

,IP Address,ip_node_id,global_ip_node_id
0,10.0.0.1,0,791567
1,10.0.0.10,1,791568
2,10.0.0.100,2,791569
3,10.0.0.101,3,791570
4,10.0.0.102,4,791571


User node ID range:
0 to 791566

IP node ID range:
791567 to 1524947


In [6]:
org1_edges = duckdb.sql(f"""
    WITH unique_pairs AS (
        SELECT DISTINCT
            "User ID",
            "IP Address"

        FROM read_parquet(
            '{ORG1_PATH}'
        )
    )

    SELECT
        u.user_node_id AS user_node_id,
        i.global_ip_node_id AS ip_node_id

    FROM unique_pairs p

    INNER JOIN org1_users u
        ON p."User ID" = u."User ID"

    INNER JOIN org1_ips i
        ON p."IP Address" = i."IP Address"
""").df()

print(org1_edges.shape)
display(org1_edges.head())

(1697608, 2)


,user_node_id,ip_node_id
0,695214,1348436
1,121740,1241093
2,552448,887069
3,226221,1014853
4,232256,1086796


In [7]:
print("Total edges:", len(org1_edges))

print("\nDuplicate edges:")
print(
    org1_edges.duplicated(
        subset=["user_node_id", "ip_node_id"]
    ).sum()
)

print("\nMissing values:")
print(org1_edges.isnull().sum())

print("\nUser node ID range in edges:")
print(
    org1_edges["user_node_id"].min(),
    "to",
    org1_edges["user_node_id"].max()
)

print("\nIP node ID range in edges:")
print(
    org1_edges["ip_node_id"].min(),
    "to",
    org1_edges["ip_node_id"].max()
)

print("\nUnique users represented in edges:")
print(org1_edges["user_node_id"].nunique())

print("\nUnique IPs represented in edges:")
print(org1_edges["ip_node_id"].nunique())

Total edges: 1697608

Duplicate edges:
0

Missing values:
user_node_id    0
ip_node_id      0
dtype: int64

User node ID range in edges:
0 to 791566

IP node ID range in edges:
791567 to 1524947

Unique users represented in edges:
791567

Unique IPs represented in edges:
733381


In [8]:
org1_edge_features = duckdb.sql(f"""
    WITH pair_features AS (
        SELECT
            "User ID",
            "IP Address",

            COUNT(*) AS total_interactions,

            SUM(
                CASE WHEN "Login Successful" = TRUE
                THEN 1 ELSE 0 END
            ) AS successful_interactions,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_interactions,

            AVG(
                CASE WHEN "Login Successful" = FALSE
                THEN 1.0 ELSE 0.0 END
            ) AS failure_ratio,

            MIN("Login Timestamp") AS first_seen,

            MAX("Login Timestamp") AS last_seen

        FROM read_parquet(
            '{ORG1_PATH}'
        )

        GROUP BY
            "User ID",
            "IP Address"
    )

    SELECT
        u.user_node_id,
        i.global_ip_node_id AS ip_node_id,

        p.total_interactions,
        p.successful_interactions,
        p.failed_interactions,
        p.failure_ratio,
        p.first_seen,
        p.last_seen

    FROM pair_features p

    INNER JOIN org1_users u
        ON p."User ID" = u."User ID"

    INNER JOIN org1_ips i
        ON p."IP Address" = i."IP Address"
""").df()

print(org1_edge_features.shape)
display(org1_edge_features.head())

(1697608, 8)


,user_node_id,ip_node_id,total_interactions,successful_interactions,failed_interactions,failure_ratio,first_seen,last_seen
0,670213,1493312,1,0.0,1.0,1.0,2020-02-06 15:04:37.913,2020-02-06 15:04:37.913
1,193384,795939,3,3.0,0.0,0.0,2020-02-06 15:09:39.568,2020-02-06 15:37:07.346
2,86013,1035189,14,14.0,0.0,0.0,2020-02-06 15:26:18.282,2021-01-25 12:41:28.334
3,647363,902992,1,1.0,0.0,0.0,2020-02-06 16:07:44.626,2020-02-06 16:07:44.626
4,250663,1061527,1,1.0,0.0,0.0,2020-02-06 16:20:31.064,2020-02-06 16:20:31.064


In [9]:
print("Number of unique edges:")
print(len(org1_edge_features))

print("\nTotal interactions represented:")
print(org1_edge_features["total_interactions"].sum())

print("\nSuccessful interactions:")
print(org1_edge_features["successful_interactions"].sum())

print("\nFailed interactions:")
print(org1_edge_features["failed_interactions"].sum())

print("\nSuccess + Failed:")
print(
    org1_edge_features["successful_interactions"].sum()
    + org1_edge_features["failed_interactions"].sum()
)

print("\nMissing values:")
print(org1_edge_features.isnull().sum())

print("\nInvalid timestamp order:")
print(
    (
        org1_edge_features["first_seen"]
        > org1_edge_features["last_seen"]
    ).sum()
)

Number of unique edges:
1697608

Total interactions represented:
3080150

Successful interactions:
2219740.0

Failed interactions:
860410.0

Success + Failed:
3080150.0

Missing values:
user_node_id               0
ip_node_id                 0
total_interactions         0
successful_interactions    0
failed_interactions        0
failure_ratio              0
first_seen                 0
last_seen                  0
dtype: int64

Invalid timestamp order:
0


In [10]:
org1_temporal_edge_features = duckdb.sql(f"""
    WITH pair_windows AS (
        SELECT
            "User ID",
            "IP Address",

            time_bucket(
                INTERVAL '10 minutes',
                "Login Timestamp"
            ) AS ten_minute_window,

            COUNT(*) AS interactions_in_window,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_in_window

        FROM read_parquet(
            '{ORG1_PATH}'
        )

        GROUP BY
            "User ID",
            "IP Address",
            ten_minute_window
    ),

    pair_temporal AS (
        SELECT
            "User ID",
            "IP Address",

            COUNT(*) AS active_10min_windows,

            MAX(interactions_in_window)
                AS max_interactions_10min,

            MAX(failed_in_window)
                AS max_failed_interactions_10min

        FROM pair_windows

        GROUP BY
            "User ID",
            "IP Address"
    )

    SELECT
        u.user_node_id,
        i.global_ip_node_id AS ip_node_id,

        p.active_10min_windows,
        p.max_interactions_10min,
        p.max_failed_interactions_10min

    FROM pair_temporal p

    INNER JOIN org1_users u
        ON p."User ID" = u."User ID"

    INNER JOIN org1_ips i
        ON p."IP Address" = i."IP Address"
""").df()

print(org1_temporal_edge_features.shape)
display(org1_temporal_edge_features.head())

(1697608, 5)


,user_node_id,ip_node_id,active_10min_windows,max_interactions_10min,max_failed_interactions_10min
0,85907,1461035,1,1,0.0
1,134355,1086972,1,1,0.0
2,85076,1078519,2,2,1.0
3,581247,1372812,1,1,0.0
4,487857,1061541,1,1,1.0


In [11]:
org1_final_edges = org1_edge_features.merge(
    org1_temporal_edge_features,
    on=["user_node_id", "ip_node_id"],
    how="inner",
    validate="one_to_one"
)

# Convert the first/last timestamps into a numeric duration feature
org1_final_edges["relationship_duration_hours"] = (
    (
        org1_final_edges["last_seen"]
        - org1_final_edges["first_seen"]
    ).dt.total_seconds()
    / 3600
)

print(org1_final_edges.shape)

display(org1_final_edges.head())

(1697608, 12)


,user_node_id,ip_node_id,total_interactions,successful_interactions,failed_interactions,failure_ratio,first_seen,last_seen,active_10min_windows,max_interactions_10min,max_failed_interactions_10min,relationship_duration_hours
0,670213,1493312,1,0.0,1.0,1.0,2020-02-06 15:04:37.913,2020-02-06 15:04:37.913,1,1,1.0,0.000000
1,193384,795939,3,3.0,0.0,0.0,2020-02-06 15:09:39.568,2020-02-06 15:37:07.346,3,1,0.0,0.457716
2,86013,1035189,14,14.0,0.0,0.0,2020-02-06 15:26:18.282,2021-01-25 12:41:28.334,14,1,0.0,8493.252792
3,647363,902992,1,1.0,0.0,0.0,2020-02-06 16:07:44.626,2020-02-06 16:07:44.626,1,1,0.0,0.000000
4,250663,1061527,1,1.0,0.0,0.0,2020-02-06 16:20:31.064,2020-02-06 16:20:31.064,1,1,0.0,0.000000


In [12]:
org1_user_features = duckdb.sql(f"""
    WITH user_basic AS (
        SELECT
            "User ID",

            COUNT(*) AS total_logins,

            SUM(
                CASE WHEN "Login Successful" = TRUE
                THEN 1 ELSE 0 END
            ) AS successful_logins,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_logins,

            AVG(
                CASE WHEN "Login Successful" = FALSE
                THEN 1.0 ELSE 0.0 END
            ) AS failure_ratio,

            COUNT(DISTINCT "IP Address")
                AS distinct_ips

        FROM read_parquet(
            '{ORG1_PATH}'
        )

        GROUP BY "User ID"
    ),

    user_windows AS (
        SELECT
            "User ID",

            time_bucket(
                INTERVAL '10 minutes',
                "Login Timestamp"
            ) AS ten_minute_window,

            COUNT(*) AS logins_in_window,

            COUNT(DISTINCT "IP Address")
                AS distinct_ips_in_window

        FROM read_parquet(
            '{ORG1_PATH}'
        )

        GROUP BY
            "User ID",
            ten_minute_window
    ),

    temporal_summary AS (
        SELECT
            "User ID",

            MAX(logins_in_window)
                AS max_logins_10min,

            MAX(distinct_ips_in_window)
                AS max_distinct_ips_10min

        FROM user_windows

        GROUP BY "User ID"
    )

    SELECT
        u.user_node_id,

        b.total_logins,
        b.successful_logins,
        b.failed_logins,
        b.failure_ratio,
        b.distinct_ips,

        t.max_logins_10min,
        t.max_distinct_ips_10min

    FROM user_basic b

    INNER JOIN temporal_summary t
        ON b."User ID" = t."User ID"

    INNER JOIN org1_users u
        ON b."User ID" = u."User ID"

    ORDER BY u.user_node_id
""").df()

print(org1_user_features.shape)
display(org1_user_features.head())

(791567, 8)


,user_node_id,total_logins,successful_logins,failed_logins,failure_ratio,distinct_ips,max_logins_10min,max_distinct_ips_10min
0,0,1,0.0,1.0,1.000000,1,1,1
1,1,3,2.0,1.0,0.333333,2,2,1
2,2,1,0.0,1.0,1.000000,1,1,1
3,3,1,1.0,0.0,0.000000,1,1,1
4,4,10,7.0,3.0,0.300000,1,2,1


In [13]:
print("Total user nodes:")
print(len(org1_user_features))

print("\nUnique user node IDs:")
print(org1_user_features["user_node_id"].nunique())

print("\nMissing values:")
print(org1_user_features.isnull().sum())

print("\nTotal logins represented:")
print(org1_user_features["total_logins"].sum())

print("\nSuccessful logins:")
print(org1_user_features["successful_logins"].sum())

print("\nFailed logins:")
print(org1_user_features["failed_logins"].sum())

print("\nInvalid failure ratios:")
print(
    (
        (org1_user_features["failure_ratio"] < 0)
        | (org1_user_features["failure_ratio"] > 1)
    ).sum()
)

print("\nInvalid temporal values:")
print(
    (
        org1_user_features["max_logins_10min"]
        > org1_user_features["total_logins"]
    ).sum()
)

print("\nInvalid IP diversity values:")
print(
    (
        org1_user_features["max_distinct_ips_10min"]
        > org1_user_features["distinct_ips"]
    ).sum()
)

Total user nodes:
791567

Unique user node IDs:
791567

Missing values:
user_node_id              0
total_logins              0
successful_logins         0
failed_logins             0
failure_ratio             0
distinct_ips              0
max_logins_10min          0
max_distinct_ips_10min    0
dtype: int64

Total logins represented:
3080150

Successful logins:
2219740.0

Failed logins:
860410.0

Invalid failure ratios:
0

Invalid temporal values:
0

Invalid IP diversity values:
0


In [14]:
org1_ip_features = duckdb.sql(f"""
    WITH ip_basic AS (
        SELECT
            "IP Address",

            COUNT(*) AS total_logins,

            SUM(
                CASE WHEN "Login Successful" = TRUE
                THEN 1 ELSE 0 END
            ) AS successful_logins,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_logins,

            AVG(
                CASE WHEN "Login Successful" = FALSE
                THEN 1.0 ELSE 0.0 END
            ) AS failure_ratio,

            COUNT(DISTINCT "User ID")
                AS distinct_users

        FROM read_parquet(
            '{ORG1_PATH}'
        )

        GROUP BY "IP Address"
    ),

    ip_windows AS (
        SELECT
            "IP Address",

            time_bucket(
                INTERVAL '10 minutes',
                "Login Timestamp"
            ) AS ten_minute_window,

            COUNT(*) AS logins_in_window,

            COUNT(DISTINCT "User ID")
                AS distinct_users_in_window

        FROM read_parquet(
            '{ORG1_PATH}'
        )

        GROUP BY
            "IP Address",
            ten_minute_window
    ),

    temporal_summary AS (
        SELECT
            "IP Address",

            MAX(logins_in_window)
                AS max_logins_10min,

            MAX(distinct_users_in_window)
                AS max_distinct_users_10min

        FROM ip_windows

        GROUP BY "IP Address"
    )

    SELECT
        i.global_ip_node_id AS ip_node_id,

        b.total_logins,
        b.successful_logins,
        b.failed_logins,
        b.failure_ratio,
        b.distinct_users,

        t.max_logins_10min,
        t.max_distinct_users_10min

    FROM ip_basic b

    INNER JOIN temporal_summary t
        ON b."IP Address" = t."IP Address"

    INNER JOIN org1_ips i
        ON b."IP Address" = i."IP Address"

    ORDER BY i.global_ip_node_id
""").df()

print(org1_ip_features.shape)
display(org1_ip_features.head())

(733381, 8)


,ip_node_id,total_logins,successful_logins,failed_logins,failure_ratio,distinct_users,max_logins_10min,max_distinct_users_10min
0,791567,25,16.0,9.0,0.360000,22,2,1
1,791568,22,19.0,3.0,0.136364,15,3,1
2,791569,92,52.0,40.0,0.434783,68,3,1
3,791570,67,33.0,34.0,0.507463,49,3,1
4,791571,88,41.0,47.0,0.534091,65,4,2


In [15]:
print("Total IP nodes:")
print(len(org1_ip_features))

print("\nUnique IP node IDs:")
print(org1_ip_features["ip_node_id"].nunique())

print("\nMissing values:")
print(org1_ip_features.isnull().sum())

print("\nTotal logins represented:")
print(org1_ip_features["total_logins"].sum())

print("\nSuccessful logins:")
print(org1_ip_features["successful_logins"].sum())

print("\nFailed logins:")
print(org1_ip_features["failed_logins"].sum())

print("\nInvalid failure ratios:")
print(
    (
        (org1_ip_features["failure_ratio"] < 0)
        | (org1_ip_features["failure_ratio"] > 1)
    ).sum()
)

print("\nInvalid temporal values:")
print(
    (
        org1_ip_features["max_logins_10min"]
        > org1_ip_features["total_logins"]
    ).sum()
)

print("\nInvalid user diversity values:")
print(
    (
        org1_ip_features["max_distinct_users_10min"]
        > org1_ip_features["distinct_users"]
    ).sum()
)

print("\nIP node ID range:")
print(
    org1_ip_features["ip_node_id"].min(),
    "to",
    org1_ip_features["ip_node_id"].max()
)

Total IP nodes:
733381

Unique IP node IDs:
733381

Missing values:
ip_node_id                  0
total_logins                0
successful_logins           0
failed_logins               0
failure_ratio               0
distinct_users              0
max_logins_10min            0
max_distinct_users_10min    0
dtype: int64

Total logins represented:
3080150

Successful logins:
2219740.0

Failed logins:
860410.0

Invalid failure ratios:
0

Invalid temporal values:
0

Invalid user diversity values:
0

IP node ID range:
791567 to 1524947


In [16]:
import pandas as pd

user_node_table = org1_user_features.rename(
    columns={
        "distinct_ips": "distinct_neighbors",
        "max_distinct_ips_10min": "max_distinct_neighbors_10min"
    }
).copy()

user_node_table["node_type"] = 0   # 0 = User


ip_node_table = org1_ip_features.rename(
    columns={
        "ip_node_id": "node_id",
        "distinct_users": "distinct_neighbors",
        "max_distinct_users_10min": "max_distinct_neighbors_10min"
    }
).copy()

ip_node_table["node_type"] = 1     # 1 = IP


user_node_table = user_node_table.rename(
    columns={"user_node_id": "node_id"}
)


org1_node_features = pd.concat(
    [user_node_table, ip_node_table],
    ignore_index=True
).sort_values("node_id").reset_index(drop=True)


print(org1_node_features.shape)
display(org1_node_features.head())
display(org1_node_features.tail())

(1524948, 9)


,node_id,total_logins,successful_logins,failed_logins,failure_ratio,distinct_neighbors,max_logins_10min,max_distinct_neighbors_10min,node_type
0,0,1,0.0,1.0,1.000000,1,1,1,0
1,1,3,2.0,1.0,0.333333,2,2,1,0
2,2,1,0.0,1.0,1.000000,1,1,1,0
3,3,1,1.0,0.0,0.000000,1,1,1,0
4,4,10,7.0,3.0,0.300000,1,2,1,0


,node_id,total_logins,successful_logins,failed_logins,failure_ratio,distinct_neighbors,max_logins_10min,max_distinct_neighbors_10min,node_type
1524943,1524943,1,1.0,0.0,0.000000,1,1,1,1
1524944,1524944,3,2.0,1.0,0.333333,1,2,1,1
1524945,1524945,1,1.0,0.0,0.000000,1,1,1,1
1524946,1524946,1,1.0,0.0,0.000000,1,1,1,1
1524947,1524947,1,0.0,1.0,1.000000,1,1,1,1


In [17]:
print("Total nodes:")
print(len(org1_node_features))

print("\nUnique node IDs:")
print(org1_node_features["node_id"].nunique())

print("\nNode ID range:")
print(
    org1_node_features["node_id"].min(),
    "to",
    org1_node_features["node_id"].max()
)

print("\nMissing values:")
print(org1_node_features.isnull().sum())

print("\nNode type counts:")
print(org1_node_features["node_type"].value_counts().sort_index())

print("\nNode IDs correctly ordered:")
print(
    (
        org1_node_features["node_id"].to_numpy()
        == range(len(org1_node_features))
    ).all()
)

Total nodes:
1524948

Unique node IDs:
1524948

Node ID range:
0 to 1524947

Missing values:
node_id                         0
total_logins                    0
successful_logins               0
failed_logins                   0
failure_ratio                   0
distinct_neighbors              0
max_logins_10min                0
max_distinct_neighbors_10min    0
node_type                       0
dtype: int64

Node type counts:
node_type
0    791567
1    733381
Name: count, dtype: int64

Node IDs correctly ordered:
True


In [18]:
import numpy as np

org1_model_nodes = org1_node_features.copy()

count_features = [
    "total_logins",
    "successful_logins",
    "failed_logins",
    "distinct_neighbors",
    "max_logins_10min",
    "max_distinct_neighbors_10min"
]

for column in count_features:
    org1_model_nodes[column] = np.log1p(
        org1_model_nodes[column]
    )

display(org1_model_nodes.head())

,node_id,total_logins,successful_logins,failed_logins,failure_ratio,distinct_neighbors,max_logins_10min,max_distinct_neighbors_10min,node_type
0,0,0.693147,0.000000,0.693147,1.000000,0.693147,0.693147,0.693147,0
1,1,1.386294,1.098612,0.693147,0.333333,1.098612,1.098612,0.693147,0
2,2,0.693147,0.000000,0.693147,1.000000,0.693147,0.693147,0.693147,0
3,3,0.693147,0.693147,0.000000,0.000000,0.693147,0.693147,0.693147,0
4,4,2.397895,2.079442,1.386294,0.300000,0.693147,1.098612,0.693147,0


In [19]:
print("Missing values:")
print(org1_model_nodes.isnull().sum())

print("\nInfinite values:")
print(
    np.isinf(
        org1_model_nodes[
            count_features
        ].to_numpy()
    ).sum()
)

print("\nFailure ratio range:")
print(
    org1_model_nodes["failure_ratio"].min(),
    "to",
    org1_model_nodes["failure_ratio"].max()
)

print("\nNode type values:")
print(
    sorted(
        org1_model_nodes["node_type"].unique()
    )
)

print("\nNode IDs unchanged:")
print(
    (
        org1_model_nodes["node_id"]
        == org1_node_features["node_id"]
    ).all()
)

Missing values:
node_id                         0
total_logins                    0
successful_logins               0
failed_logins                   0
failure_ratio                   0
distinct_neighbors              0
max_logins_10min                0
max_distinct_neighbors_10min    0
node_type                       0
dtype: int64

Infinite values:
0

Failure ratio range:
0.0 to 1.0

Node type values:
[np.int64(0), np.int64(1)]

Node IDs unchanged:
True


In [20]:
model_feature_columns = [
    "total_logins",
    "successful_logins",
    "failed_logins",
    "failure_ratio",
    "distinct_neighbors",
    "max_logins_10min",
    "max_distinct_neighbors_10min",
    "node_type"
]

x_numpy = org1_model_nodes[
    model_feature_columns
].to_numpy(
    dtype=np.float32
)

print("Feature matrix shape:")
print(x_numpy.shape)

print("\nData type:")
print(x_numpy.dtype)

print("\nMemory usage (MB):")
print(x_numpy.nbytes / (1024 ** 2))

print("\nAll values finite:")
print(np.isfinite(x_numpy).all())

print("\nFirst node feature vector:")
print(x_numpy[0])

Feature matrix shape:
(1524948, 8)

Data type:
float32

Memory usage (MB):
46.5377197265625

All values finite:
True

First node feature vector:
[0.6931472 0.        0.6931472 1.        0.6931472 0.6931472 0.6931472
 0.       ]


In [21]:
# Extract the two ends of every structural User-IP edge
user_nodes = org1_final_edges[
    "user_node_id"
].to_numpy(dtype=np.int64)

ip_nodes = org1_final_edges[
    "ip_node_id"
].to_numpy(dtype=np.int64)

# Add both directions so the graph behaves as undirected
edge_index_numpy = np.vstack([
    np.concatenate([user_nodes, ip_nodes]),
    np.concatenate([ip_nodes, user_nodes])
])

print("Edge index shape:")
print(edge_index_numpy.shape)

print("\nData type:")
print(edge_index_numpy.dtype)

print("\nDirected edges:")
print(edge_index_numpy.shape[1])

print("\nOriginal structural edges:")
print(len(org1_final_edges))

print("\nNode ID range in edge_index:")
print(
    edge_index_numpy.min(),
    "to",
    edge_index_numpy.max()
)

print("\nSelf-loops:")
print(
    (edge_index_numpy[0] == edge_index_numpy[1]).sum()
)

print("\nMemory usage (MB):")
print(edge_index_numpy.nbytes / (1024 ** 2))

Edge index shape:
(2, 3395216)

Data type:
int64

Directed edges:
3395216

Original structural edges:
1697608

Node ID range in edge_index:
0 to 1524947

Self-loops:
0

Memory usage (MB):
51.806884765625


In [22]:
E = len(org1_final_edges)

forward_edges = edge_index_numpy[:, :E]
reverse_edges = edge_index_numpy[:, E:]

print("Forward edge count:")
print(forward_edges.shape[1])

print("\nReverse edge count:")
print(reverse_edges.shape[1])

print("\nForward users match reverse destinations:")
print(
    np.array_equal(
        forward_edges[0],
        reverse_edges[1]
    )
)

print("\nForward IPs match reverse sources:")
print(
    np.array_equal(
        forward_edges[1],
        reverse_edges[0]
    )
)

print("\nExactly twice the structural edge count:")
print(
    edge_index_numpy.shape[1]
    == 2 * len(org1_final_edges)
)

Forward edge count:
1697608

Reverse edge count:
1697608

Forward users match reverse destinations:
True

Forward IPs match reverse sources:
True

Exactly twice the structural edge count:
True


In [23]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python executable:
c:\Users\Gulso\Documents\Gul\CredBlock-FL\.venv\Scripts\python.exe

Python version:
3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]


In [24]:
import torch

x = torch.from_numpy(x_numpy)

edge_index = torch.from_numpy(edge_index_numpy)

print("Node feature tensor:")
print("Shape:", x.shape)
print("Data type:", x.dtype)
print(
    "Memory (MB):",
    x.element_size() * x.nelement() / (1024 ** 2)
)

print("\nEdge index tensor:")
print("Shape:", edge_index.shape)
print("Data type:", edge_index.dtype)
print(
    "Memory (MB):",
    edge_index.element_size()
    * edge_index.nelement()
    / (1024 ** 2)
)

print("\nFeature values finite:")
print(torch.isfinite(x).all().item())

print("\nEdge node ID range:")
print(
    edge_index.min().item(),
    "to",
    edge_index.max().item()
)

Node feature tensor:
Shape: torch.Size([1524948, 8])
Data type: torch.float32
Memory (MB): 46.5377197265625

Edge index tensor:
Shape: torch.Size([2, 3395216])
Data type: torch.int64
Memory (MB): 51.806884765625

Feature values finite:
True

Edge node ID range:
0 to 1524947


In [26]:
from pathlib import Path
import torch

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

torch.save(x, output_dir / "org1_node_features.pt")
torch.save(edge_index, output_dir / "org1_edge_index.pt")

print("Saved:")
print(output_dir / "org1_node_features.pt")
print(output_dir / "org1_edge_index.pt")

Saved:
..\data\processed\org1_node_features.pt
..\data\processed\org1_edge_index.pt


In [27]:
x_loaded = torch.load(output_dir / "org1_node_features.pt")
edge_index_loaded = torch.load(output_dir / "org1_edge_index.pt")

print("Loaded node tensor:", x_loaded.shape, x_loaded.dtype)
print("Loaded edge tensor:", edge_index_loaded.shape, edge_index_loaded.dtype)

print("Node tensor matches:", torch.equal(x, x_loaded))
print("Edge tensor matches:", torch.equal(edge_index, edge_index_loaded))

Loaded node tensor: torch.Size([1524948, 8]) torch.float32
Loaded edge tensor: torch.Size([2, 3395216]) torch.int64
Node tensor matches: True
Edge tensor matches: True


In [29]:
import torch_geometric

print("PyTorch Geometric version:", torch_geometric.__version__)

c:\Users\Gulso\Documents\Gul\CredBlock-FL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Gulso\Documents\Gul\CredBlock-FL\.venv\Lib\site-packages\torch\jit\_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


PyTorch Geometric version: 2.8.0.post1


In [30]:
from torch_geometric.data import Data

org1_graph = Data(
    x=x,
    edge_index=edge_index
)

print(org1_graph)

print("\nGraph summary:")
print("Number of nodes:", org1_graph.num_nodes)
print("Number of edges:", org1_graph.num_edges)
print("Number of node features:", org1_graph.num_node_features)

print("\nGraph validation:")
print(org1_graph.validate(raise_on_error=False))

Data(x=[1524948, 8], edge_index=[2, 3395216])

Graph summary:
Number of nodes: 1524948
Number of edges: 3395216
Number of node features: 8

Graph validation:
True


In [31]:
torch.save(org1_graph, output_dir / "org1_graph.pt")

print("Saved:", output_dir / "org1_graph.pt")

Saved: ..\data\processed\org1_graph.pt


In [32]:
loaded_graph = torch.load(
    output_dir / "org1_graph.pt",
    weights_only=False
)

print(loaded_graph)
print("Nodes:", loaded_graph.num_nodes)
print("Edges:", loaded_graph.num_edges)
print("Features:", loaded_graph.num_node_features)
print("Validation:", loaded_graph.validate(raise_on_error=False))

Data(x=[1524948, 8], edge_index=[2, 3395216])
Nodes: 1524948
Edges: 3395216
Features: 8
Validation: True
